# Evaluation & Forecasting — Wildfire Risk Forecaster

SHAP analysis for model interpretability, Prophet time-series forecasting,
and seasonal risk projections.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from prophet import Prophet
import joblib

## Evaluation Steps

1. Load trained XGBoost model and test data
2. Compute SHAP values for feature importance interpretation
3. Build Prophet forecast of monthly fire risk index
4. Generate seasonal risk projections and confidence intervals

In [ ]:
# SHAP analysis
model = joblib.load('../models/xgb_wildfire.pkl')
X_test = pd.read_parquet('../data/processed/X_test.parquet')

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig('../reports/shap_summary.png', dpi=150)
plt.show()

In [ ]:
# Prophet time-series forecasting
risk_ts = pd.read_csv('../data/processed/monthly_risk_index.csv')
risk_ts.columns = ['ds', 'y']

m = Prophet(yearly_seasonality=True, weekly_seasonality=False)
m.fit(risk_ts)

future = m.make_future_dataframe(periods=12, freq='MS')
forecast = m.predict(future)

fig = m.plot(forecast)
plt.title('Wildfire Risk Index — 12-Month Forecast')
plt.ylabel('Risk Index')
plt.tight_layout()
plt.show()